[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-ramakrishnan/vision_transformers/blob/main/autoencoder_vae.ipynb)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")

In [ ]:
BATCH_SIZE = 128

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# =====================================================================
# PART 1: Vanilla Autoencoder
# =====================================================================
class Autoencoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, bottleneck_dim=64, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim),
            nn.ReLU(),
            nn.Linear(bottleneck_dim, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, bottleneck_dim),
            nn.ReLU(),
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        b = x.size(0)
        x_flat = x.view(b, -1)
        z = self.encode(x_flat)
        x_recon = self.decode(z)
        return x_recon.view(b, 1, 28, 28), z

In [ ]:
AE_EPOCHS = 10
AE_LR = 1e-3
AE_LATENT_DIM = 2

ae_model = Autoencoder(latent_dim=AE_LATENT_DIM).to(device)
ae_criterion = nn.MSELoss()
ae_optimizer = optim.Adam(ae_model.parameters(), lr=AE_LR)

for epoch in range(AE_EPOCHS):
    ae_model.train()
    total_loss = 0.0
    for images, _ in train_loader:
        images = images.to(device)
        recon_images, _ = ae_model(images)
        loss = ae_criterion(recon_images, images)

        ae_optimizer.zero_grad()
        loss.backward()
        ae_optimizer.step()
        total_loss += loss.item() * images.size(0)

    avg_loss = total_loss / len(train_loader.dataset)
    print(f"[Autoencoder] Epoch [{epoch + 1}/{AE_EPOCHS}] - MSE Loss: {avg_loss:.6f}")

In [ ]:
ae_model.eval()
with torch.no_grad():
    test_images, _ = next(iter(test_loader))
    test_images = test_images[:8].to(device)
    recon_images, _ = ae_model(test_images)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(test_images[i].cpu().squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon_images[i].cpu().squeeze(), cmap="gray")
    axes[1, i].axis("off")
plt.suptitle("Vanilla Autoencoder: Original (Top) vs Reconstructed (Bottom)")
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# PART 2: Variational Autoencoder (VAE)
# =====================================================================
class Encoder(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, latent_dim=20, hidden_dim=400, output_dim=784):
        super().__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, z):
        h = F.relu(self.fc1(z))
        return torch.sigmoid(self.fc2(h))


class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        b = x.size(0)
        x_flat = x.view(b, -1)
        mu, logvar = self.encoder(x_flat)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decoder(z)
        return x_recon, mu, logvar


def vae_loss_function(recon_x, x, mu, logvar):
    x_flat = x.view(x.size(0), -1)
    bce = F.binary_cross_entropy(recon_x, x_flat, reduction="sum")
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + kld, bce, kld

In [ ]:
VAE_EPOCHS = 15
VAE_LR = 1e-3
VAE_LATENT_DIM = 20

vae_model = VAE(input_dim=784, hidden_dim=400, latent_dim=VAE_LATENT_DIM).to(device)
vae_optimizer = optim.Adam(vae_model.parameters(), lr=VAE_LR)

for epoch in range(VAE_EPOCHS):
    vae_model.train()
    total_loss = 0.0
    total_bce = 0.0
    total_kld = 0.0

    for images, _ in train_loader:
        images = images.to(device)
        recon_images, mu, logvar = vae_model(images)
        loss, bce, kld = vae_loss_function(recon_images, images, mu, logvar)

        vae_optimizer.zero_grad()
        loss.backward()
        vae_optimizer.step()

        total_loss += loss.item()
        total_bce += bce.item()
        total_kld += kld.item()

    n_samples = len(train_loader.dataset)
    print(
        f"[VAE] Epoch [{epoch + 1}/{VAE_EPOCHS}] - "
        f"Total Loss: {total_loss / n_samples:.4f} | "
        f"BCE: {total_bce / n_samples:.4f} | "
        f"KLD: {total_kld / n_samples:.4f}"
    )

In [ ]:
vae_model.eval()
with torch.no_grad():
    # 1. Reconstructions
    test_images, _ = next(iter(test_loader))
    test_images = test_images[:8].to(device)
    recon_flat, _, _ = vae_model(test_images)
    recon_images = recon_flat.view(-1, 1, 28, 28)

    # 2. Random samples from standard normal prior z ~ N(0, I)
    z_random = torch.randn(16, VAE_LATENT_DIM, device=device)
    generated_images = vae_model.decoder(z_random).view(-1, 1, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(test_images[i].cpu().squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon_images[i].cpu().squeeze(), cmap="gray")
    axes[1, i].axis("off")
plt.suptitle("VAE: Original (Top) vs Reconstructed (Bottom)")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(16):
    r, c = divmod(i, 8)
    axes[r, c].imshow(generated_images[i].cpu().squeeze(), cmap="gray")
    axes[r, c].axis("off")
plt.suptitle("VAE: Generated Digits Sampled from Prior z ~ N(0, I)")
plt.tight_layout()
plt.show()